# Gikomba Market Stall Performance Analysis

**Author:** Asman Makero Musa · PLP Data Analytics Cohort 7  
**Context:** Freelance Data Analyst Evaluation for Wanjiku's Stall at Gikomba Market  

### Executive Summary & Problem Statement
Wanjiku operates a fresh-produce stall in Nairobi's busy Gikomba Market. Over the past 14 days, she has logged her daily sales in an exercise book while monitoring stock levels across five core staples. Seeking to make data-informed operational decisions, she hired an analyst to answer key business questions: How is the business performing overall? Is it growing week-on-week? Which items require restocking, and where should managerial attention be focused? 

Using pure core Python (with zero external dependencies), this analysis audits daily revenue, isolates growth trends and day-of-week patterns, flags inventory reorder risks, and provides concrete, evidence-backed recommendations.

### Dataset Initialization
We initialize the raw tracking data recorded by Wanjiku over the 14-day observation period, consisting of daily sales in Kenyan Shillings (KES), repeated 7-day cyclical labels, and current inventory metrics (stock quantity, reorder threshold, procurement cost, and retail selling price per kilogram).

In [1]:
# Raw dataset provided for Wanjiku's stall
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Overall Sales Metrics and Best/Worst Days

**Approach:**  
We calculate the aggregate sales revenue using `sum(daily_sales)`, the overall 14-day mean using standard arithmetic (`sum / len`), and the peak and trough revenue values using `max()` and `min()`. To determine the corresponding day name and week number, we use `daily_sales.index()` to retrieve zero-based list positions. Indices 0 to 6 belong to Week 1, while indices 7 to 13 correspond to Week 2.

In [2]:
# Task 1: Overall sales metrics, best and worst days
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_idx = daily_sales.index(highest_sales)
worst_idx = daily_sales.index(lowest_sales)

best_day_name = days[best_idx]
best_week_num = 1 if best_idx < 7 else 2

worst_day_name = days[worst_idx]
worst_week_num = 1 if worst_idx < 7 else 2

print(f"Total Sales (14 Days) : KES {total_sales:,.2f}")
print(f"Average Daily Sales   : KES {average_sales:,.2f}")
print(f"Highest Daily Sales   : KES {highest_sales:,.2f} on {best_day_name} (Week {best_week_num}, Day {best_idx + 1})")
print(f"Lowest Daily Sales    : KES {lowest_sales:,.2f} on {worst_day_name} (Week {worst_week_num}, Day {worst_idx + 1})")

Total Sales (14 Days) : KES 46,450.00
Average Daily Sales   : KES 3,317.86
Highest Daily Sales   : KES 5,600.00 on Sat (Week 2, Day 13)
Lowest Daily Sales    : KES 1,950.00 on Tue (Week 1, Day 2)


## Task 2: Weekly Totals and Week-on-Week Growth

**Approach:**  
We define a modular function `week_totals(sales)` that partitions the 14-day list into two 7-day slices (`sales[:7]` and `sales[7:]`) and returns the sum of each week packed inside a tuple. We unpack the tuple into `w1_total` and `w2_total` and calculate the Week-on-Week (WoW) percentage growth using the relative change formula: `((w2_total - w1_total) / w1_total) * 100`.

In [3]:
def week_totals(sales):
    """
    Calculate and return the total sales for Week 1 and Week 2 as a tuple.
    
    Parameters:
        sales (list): A list of 14 numerical daily sales values.
        
    Returns:
        tuple: (week_1_total, week_2_total)
    """
    week_1_total = sum(sales[:7])
    week_2_total = sum(sales[7:])
    return week_1_total, week_2_total

# Calculate weekly totals and WoW growth
w1_total, w2_total = week_totals(daily_sales)
wow_growth_pct = ((w2_total - w1_total) / w1_total) * 100

print(f"Week 1 Total Sales : KES {w1_total:,.2f}")
print(f"Week 2 Total Sales : KES {w2_total:,.2f}")
print(f"Week-on-Week Growth: {wow_growth_pct:+.2f}%")

Week 1 Total Sales : KES 22,050.00
Week 2 Total Sales : KES 24,400.00
Week-on-Week Growth: +10.66%


## Task 3: Day Classification and Category Frequency

**Approach:**  
We construct a reusable function `classify_day(amount, average)` that applies business rules:
- **Strong**: Sales exceeding 20% above average (`amount > average * 1.20`)
- **Weak**: Sales falling more than 20% below average (`amount < average * 0.80`)
- **Normal**: Sales within the $\pm 20\%$ corridor.

We then iterate through all 14 days and record the frequency of each category using the dictionary `.get()` accumulator pattern (`counts[label] = counts.get(label, 0) + 1`).

In [4]:
def classify_day(amount, average):
    """
    Classify sales performance of a given day relative to the period average.
    
    Parameters:
        amount (float or int): Daily sales figure.
        average (float or int): Benchmark average daily sales.
        
    Returns:
        str: 'Strong' (>20% above avg), 'Weak' (<20% below avg), or 'Normal'.
    """
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    else:
        return "Normal"

# Evaluate thresholds
upper_threshold = average_sales * 1.20
lower_threshold = average_sales * 0.80

# Classify all days and count occurrences
category_counts = {}
daily_records = []

print(f"Upper Threshold (> 20% above avg): KES {upper_threshold:,.2f}")
print(f"Lower Threshold (< 20% below avg): KES {lower_threshold:,.2f}\n")
print("Day-by-Day Classifications:")

for idx, sale in enumerate(daily_sales):
    cat = classify_day(sale, average_sales)
    daily_records.append((idx + 1, days[idx], sale, cat))
    category_counts[cat] = category_counts.get(cat, 0) + 1
    print(f"  Day {idx + 1:>2} ({days[idx]}): KES {sale:,} -> {cat}")

print("\nCategory Summary:")
for category in ["Weak", "Normal", "Strong"]:
    print(f"  - {category:<7}: {category_counts.get(category, 0)} days")

Upper Threshold (> 20% above avg): KES 3,981.43
Lower Threshold (< 20% below avg): KES 2,654.29

Day-by-Day Classifications:
  Day  1 (Mon): KES 2,300 -> Weak
  Day  2 (Tue): KES 1,950 -> Weak
  Day  3 (Wed): KES 2,800 -> Normal
  Day  4 (Thu): KES 2,100 -> Weak
  Day  5 (Fri): KES 3,600 -> Normal
  Day  6 (Sat): KES 5,200 -> Strong
  Day  7 (Sun): KES 4,100 -> Strong
  Day  8 (Mon): KES 2,500 -> Weak
  Day  9 (Tue): KES 2,200 -> Weak
  Day 10 (Wed): KES 3,100 -> Normal
  Day 11 (Thu): KES 2,400 -> Weak
  Day 12 (Fri): KES 3,900 -> Normal
  Day 13 (Sat): KES 5,600 -> Strong
  Day 14 (Sun): KES 4,700 -> Strong

Category Summary:
  - Weak   : 6 days
  - Normal : 4 days
  - Strong : 4 days


## Task 4: Weekdays vs. Weekends Performance Analysis

**Approach:**  
We segment the 14-day dataset into Weekday records (Monday through Friday, 10 days total) and Weekend records (Saturday and Sunday, 4 days total). We compute the mean revenue for both cohorts and evaluate the percentage difference that weekend trade delivers over weekday trade: `((weekend_avg - weekday_avg) / weekday_avg) * 100`.

In [5]:
# Task 4: Weekday vs Weekend Comparison
weekday_sales = [sale for i, sale in enumerate(daily_sales) if days[i] in ["Mon", "Tue", "Wed", "Thu", "Fri"]]
weekend_sales = [sale for i, sale in enumerate(daily_sales) if days[i] in ["Sat", "Sun"]]

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium = ((weekend_avg - weekday_avg) / weekday_avg) * 100

print(f"Weekday Average Sales (10 days) : KES {weekday_avg:,.2f}")
print(f"Weekend Average Sales (4 days)  : KES {weekend_avg:,.2f}")
print(f"Weekend vs. Weekday Difference   : {weekend_premium:+.2f}% higher on weekends")

Weekday Average Sales (10 days) : KES 2,685.00
Weekend Average Sales (4 days)  : KES 4,900.00
Weekend vs. Weekday Difference   : +82.50% higher on weekends


## Task 5: Inventory Reorder Evaluation and Gross Margins

**Approach:**  
We define a function `check_reorder_levels(inventory_data)` that iterates over the stock records with `.items()` and isolates products whose current stock is less than or equal to their safety reorder threshold (`qty <= reorder`). Furthermore, we loop through each commodity to compute the gross profit margin percentage using the commercial formula: `((price - cost) / price) * 100`.

In [6]:
def check_reorder_levels(inventory_data):
    """
    Identify commodities in the inventory whose current stock is at or below the reorder point.
    
    Parameters:
        inventory_data (dict): Nested dictionary with item stats (qty, reorder, cost, price).
        
    Returns:
        list: Names of items needing urgent replenishment.
    """
    items_to_reorder = []
    for item_name, details in inventory_data.items():
        if details["qty"] <= details["reorder"]:
            items_to_reorder.append(item_name)
    return items_to_reorder

# Run reorder check and calculate margins
reorder_needed = check_reorder_levels(stock)

margins = {}
print(f"Products Requiring Immediate Reorder: {reorder_needed}\n")
print("Product Profitability & Margins:")
for item, details in stock.items():
    margin = ((details["price"] - details["cost"]) / details["price"]) * 100
    margins[item] = margin
    print(f"  - {item:<9}: {margin:.1f}% gross margin (Cost: KES {details['cost']:>3}, Price: KES {details['price']:>3})")

Products Requiring Immediate Reorder: ['Tomatoes', 'Sukuma']

Product Profitability & Margins:
  - Tomatoes : 33.3% gross margin (Cost: KES  60, Price: KES  90)
  - Onions   : 30.0% gross margin (Cost: KES  70, Price: KES 100)
  - Sukuma   : 40.0% gross margin (Cost: KES  30, Price: KES  50)
  - Potatoes : 33.3% gross margin (Cost: KES  50, Price: KES  75)
  - Avocado  : 42.9% gross margin (Cost: KES  80, Price: KES 140)


## Task 6: Stall Performance Report

**Approach:**  
We construct and display an executive 'Stall Performance Report' utilizing Python f-strings with proper column alignment, structured visual delimiters, and currency figures formatted with thousands separators (`:,` or `:,.2f`).

In [7]:
# Task 6: Stall Performance Report
banner = "=" * 62
divider = "-" * 62

print(banner)
print("                WANJIKU'S GIKOMBA MARKET STALL                ")
print("             TWO-WEEK COMPREHENSIVE AUDIT REPORT              ")
print(banner)
print("FINANCIAL OVERVIEW:")
print(f"  Total 14-Day Sales Revenue : KES {total_sales:>13,.2f}")
print(f"  Overall Daily Average      : KES {average_sales:>13,.2f}")
print(f"  Week 1 Total Revenue       : KES {w1_total:>13,.2f}")
print(f"  Week 2 Total Revenue       : KES {w2_total:>13,.2f}")
print(f"  Week-on-Week Growth Rate   : {wow_growth_pct:>18.2f}%")
print(divider)
print("DAILY HIGHS & LOWS:")
print(f"  Highest Sales Day          : {best_day_name} (Week {best_week_num}, Day {best_idx + 1}) - KES {highest_sales:,}")
print(f"  Lowest Sales Day           : {worst_day_name} (Week {worst_week_num}, Day {worst_idx + 1})  - KES {lowest_sales:,}")
print(divider)
print("DAY-TYPE COMPARISON:")
print(f"  Weekday Average (Mon - Fri): KES {weekday_avg:>13,.2f}  (10 days)")
print(f"  Weekend Average (Sat - Sun): KES {weekend_avg:>13,.2f}  (4 days)")
print(f"  Weekend Sales Surge        : {weekend_premium:>18.2f}% vs weekdays")
print(divider)
print("DAY CLASSIFICATION BREAKDOWN (±20% Average Corridor):")
print(f"  • Strong Days (> KES {upper_threshold:,.0f}):  {category_counts.get('Strong', 0)} days ({(category_counts.get('Strong', 0)/14)*100:.1f}%)")
print(f"  • Normal Days (KES {lower_threshold:,.0f} - {upper_threshold:,.0f}):  {category_counts.get('Normal', 0)} days ({(category_counts.get('Normal', 0)/14)*100:.1f}%)")
print(f"  • Weak Days   (< KES {lower_threshold:,.0f}):  {category_counts.get('Weak', 0)} days ({(category_counts.get('Weak', 0)/14)*100:.1f}%)")
print(divider)
print("INVENTORY & REORDER AUDIT:")
print(f"{'Item':<12} {'Stock (kg)':>10} {'Reorder (kg)':>14} {'Margin (%)':>12}   {'Restock Status'}")
print(divider)
for item, details in stock.items():
    status = "CRITICAL REORDER" if item in reorder_needed else "Adequate"
    margin = margins[item]
    print(f"{item:<12} {details['qty']:>10} {details['reorder']:>14} {margin:>11.1f}%   {status}")
print(banner)

                WANJIKU'S GIKOMBA MARKET STALL                
             TWO-WEEK COMPREHENSIVE AUDIT REPORT              
FINANCIAL OVERVIEW:
  Total 14-Day Sales Revenue : KES     46,450.00
  Overall Daily Average      : KES      3,317.86
  Week 1 Total Revenue       : KES     22,050.00
  Week 2 Total Revenue       : KES     24,400.00
  Week-on-Week Growth Rate   :              10.66%
--------------------------------------------------------------
DAILY HIGHS & LOWS:
  Highest Sales Day          : Sat (Week 2, Day 13) - KES 5,600
  Lowest Sales Day           : Tue (Week 1, Day 2)  - KES 1,950
--------------------------------------------------------------
DAY-TYPE COMPARISON:
  Weekday Average (Mon - Fri): KES      2,685.00  (10 days)
  Weekend Average (Sat - Sun): KES      4,900.00  (4 days)
  Weekend Sales Surge        :              82.50% vs weekdays
--------------------------------------------------------------
DAY CLASSIFICATION BREAKDOWN (±20% Average Corridor):
  • Strong Da

## Task 7: Recommendations for Wanjiku

Based on the empirical findings from the 14-day analysis, here are four concrete, data-backed recommendations for Wanjiku's stall:

1. **Pre-Weekend Stocking for High Traffic Volume**: Weekend sales average **KES 4,900.00**, representing an **82.50% surge** above the weekday average of **KES 2,685.00**, peaking on Saturday of Week 2 at **KES 5,600.00**. Wanjiku must order extra produce on Friday mornings to ensure her inventory does not stock out during Saturday and Sunday peak shopping hours.
2. **Immediate Replenishment of Tomatoes and Sukuma**: Both **Tomatoes** (18 kg on hand vs. 25 kg reorder threshold) and **Sukuma** (9 kg on hand vs. 15 kg reorder threshold) have dropped into critical deficit. Sukuma yields a high gross margin of **40.0%** and Tomatoes yield **33.3%**; failing to restock them immediately risks lost revenue and customer defection on essential daily staples.
3. **Midweek Bundle Promotions to Lift Weak Days**: A total of **6 out of 14 days (42.9%)** were classified as **Weak** (earning below KES 2,654.29, with Tuesday of Week 1 hitting a low of **KES 1,950.00**). To boost weekday footfall, Wanjiku should introduce promotional Tuesday-Thursday meal bundles (e.g., pairing high-margin Avocado at **42.9%** and Sukuma at **40.0%** with Onions and Potatoes).
4. **Reinvest Week-on-Week Growth into Safety Buffers**: Sales grew **+10.66%** in Week 2 (from KES 22,050.00 to KES 24,400.00), generating an additional KES 2,350.00. Wanjiku should allocate this incremental cash flow to maintain sufficient buffer stock for high-turnover items rather than allowing cash to sit idle.